# Day 5: Retrieval, Walked Through

This notebook ties together Days 1-4 into a working retrieval system, then goes
further: query preprocessing, quality metrics, multi-query search, hybrid search,
and the handoff into ranking.

Run each cell in order. Plain Python throughout -- self-contained, so it doesn't
depend on importing the other day folders.

## Cell 1: What is retrieval and why it matters

Retrieval is the part of RAG that finds the right documents for a question --
like a search engine, but instead of showing you a results page, the results get
handed to a language model to write an answer.

Good retrieval -> good answers. Bad retrieval -> bad answers, no matter how capable
the language model is. You can't generate your way out of missing information.

## Cell 2: Build a basic retriever

Let's build the core pipeline: chunk documents, embed the chunks, store them, and
search by query.

In [ ]:
import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    ma = math.sqrt(sum(x * x for x in a))
    mb = math.sqrt(sum(y * y for y in b))
    return dot / (ma * mb) if ma and mb else 0.0

# Six concept dimensions (matching Day 2's fuller vocabulary) instead of a
# handful -- too few dimensions makes cosine similarity land on a small
# set of "special angles" (like exactly 0.707) regardless of which words
# actually matched, which produces confusing, uninformative demos.
CONCEPTS = {
    "programming":   ["python", "programming", "language", "syntax", "code"],
    "data_types":    ["data", "types", "strings", "integers", "floats", "lists"],
    "retrieval":     ["rag", "retrieval", "retrieves", "generation", "lookup", "look", "find"],
    "freshness":     ["outdated", "fresh", "matters", "old", "problem", "problems"],
    "search_tech":   ["vector", "database", "embeddings", "similarity", "search", "meaning"],
    "keyword_match": ["keyword", "exact", "words", "matching", "simple", "fast"],
}

def text_to_embedding(text):
    words = set(w.strip("?.,!").lower() for w in text.split())
    return [float(sum(1 for w in words if w in cw)) for cw in CONCEPTS.values()]

def chunk_by_paragraphs(text):
    return [p.strip() for p in text.split("\n\n") if p.strip()]

class SimpleVectorDB:
    def __init__(self):
        self.entries = []
    def add(self, embedding, metadata=None):
        self.entries.append({"embedding": embedding, "metadata": metadata or {}})
    def search(self, query_embedding, top_k=3):
        scored = [(e, cosine_similarity(query_embedding, e["embedding"])) for e in self.entries]
        scored.sort(key=lambda pair: pair[1], reverse=True)
        return [{"metadata": e["metadata"], "similarity": s} for e, s in scored[:top_k]]

DOCUMENTS = [
    {"id": 2, "title": "What is RAG",
     "text": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents before generating an answer.\n\nThis approach retrieves information first, then generates a grounded response."},
    {"id": 3, "title": "What is a Vector Database",
     "text": "A vector database stores data as numerical vectors called embeddings. It allows fast similarity search.\n\nReal vector databases use indexing to keep lookups quick even at scale."},
    {"id": 5, "title": "Why RAG Matters",
     "text": "RAG matters because language models can be outdated. Retrieval lets them use fresh information.\n\nRetraining a model is slow. Updating a folder of documents is fast by comparison."},
]

class Retriever:
    def __init__(self):
        self.db = SimpleVectorDB()
    def index_documents(self, documents):
        count = 0
        for doc in documents:
            for i, chunk in enumerate(chunk_by_paragraphs(doc["text"])):
                emb = text_to_embedding(doc["title"] + " " + chunk)
                self.db.add(emb, metadata={"doc_id": doc["id"], "title": doc["title"], "chunk_index": i, "text": chunk})
                count += 1
        return count
    def retrieve(self, query, top_k=3):
        results = self.db.search(text_to_embedding(query), top_k=top_k)
        return [r for r in results if r["similarity"] > 0]

retriever = Retriever()
total = retriever.index_documents(DOCUMENTS)
print(f"Indexed {len(DOCUMENTS)} documents into {total} chunks.\n")

for r in retriever.retrieve("What is RAG?", top_k=3):
    print(f"  {r['metadata']['title']} (chunk {r['metadata']['chunk_index']}, similarity {r['similarity']:.3f})")

## Cell 3: Preprocess queries

Cleaning and expanding a query before searching can help it match documents that
used different words for the same idea.

In [ ]:
STOPWORDS = {"what", "is", "a", "an", "the", "how", "do", "i", "get"}
SYNONYMS = {"get": ["lookup", "retrieve"]}

def clean_query(query):
    query = query.lower()
    query = "".join(c for c in query if c.isalnum() or c.isspace())
    return " ".join(query.split())

def expand_query(query):
    words = query.split()
    expanded = list(words)
    for w in words:
        if w in SYNONYMS:
            expanded.extend(SYNONYMS[w])
    return " ".join(expanded)

raw_query = "How do I get the right documents?"
cleaned = clean_query(raw_query)
expanded = expand_query(cleaned)

document_snippet = "RAG retrieves relevant documents before generating an answer."

raw_sim = cosine_similarity(text_to_embedding(raw_query), text_to_embedding(document_snippet))
expanded_sim = cosine_similarity(text_to_embedding(expanded), text_to_embedding(document_snippet))

print(f"Raw query similarity:      {raw_sim:.3f}")
print(f"Expanded query similarity: {expanded_sim:.3f}")
print("\nExpanding 'get' to include 'lookup'/'retrieve' let the query match the document's own wording.")

## Cell 4: Measure quality with metrics

Precision@K, recall@K, and NDCG turn "retrieval seems fine" into an actual number.

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    if not top_k:
        return 0.0
    return sum(1 for d in top_k if d in relevant_ids) / len(top_k)

def dcg_at_k(retrieved_ids, relevance_scores, k):
    score = 0.0
    for i, doc_id in enumerate(retrieved_ids[:k]):
        score += relevance_scores.get(doc_id, 0) / math.log2(i + 2)
    return score

def ndcg_at_k(retrieved_ids, relevance_scores, k):
    actual = dcg_at_k(retrieved_ids, relevance_scores, k)
    ideal_order = sorted(relevance_scores, key=lambda d: relevance_scores[d], reverse=True)
    ideal = dcg_at_k(ideal_order, relevance_scores, k)
    return actual / ideal if ideal else 1.0

relevance_scores = {2: 2, 5: 1}
good_ranking = [2, 5, 3]
bad_ranking = [3, 3, 2]

print(f"Good ranking -- precision@3: {precision_at_k(good_ranking, set(relevance_scores), 3):.2f}, NDCG@3: {ndcg_at_k(good_ranking, relevance_scores, 3):.2f}")
print(f"Bad ranking  -- precision@3: {precision_at_k(bad_ranking, set(relevance_scores), 3):.2f}, NDCG@3: {ndcg_at_k(bad_ranking, relevance_scores, 3):.2f}")

## Cell 5: Multi-query retrieval

One phrasing of a question might miss a relevant document. Searching with several
phrasings and merging results can catch what a single query misses.

In [ ]:
query = "How do I get old problems fixed?"
variations = [query, "how do i retrieve old problems fixed", "how do i lookup old problems fixed"]

single_results = retriever.retrieve(query, top_k=3)
print("Single-query results:")
for r in single_results:
    print(f"  {r['metadata']['title']} (chunk {r['metadata']['chunk_index']}, similarity {r['similarity']:.3f})")

merged = {}
for variation in variations:
    for r in retriever.retrieve(variation, top_k=3):
        key = (r["metadata"]["doc_id"], r["metadata"]["chunk_index"])
        if key not in merged or r["similarity"] > merged[key]["similarity"]:
            merged[key] = r

print("\nMulti-query results:")
for r in sorted(merged.values(), key=lambda r: r["similarity"], reverse=True)[:3]:
    print(f"  {r['metadata']['title']} (chunk {r['metadata']['chunk_index']}, similarity {r['similarity']:.3f})")

## Cell 6: Hybrid retrieval

Combining keyword search (Day 1) with semantic search (Days 2-3) catches documents
that either method alone would miss.

In [ ]:
def keyword_search_chunks(query, chunks, top_k=3):
    query_words = [w.strip("?.,!") for w in query.lower().split() if w not in STOPWORDS]
    scored = []
    for chunk in chunks:
        chunk_words = set(w.strip("?.,!") for w in chunk["text"].lower().split())
        count = sum(1 for w in query_words if w in chunk_words)
        if count > 0:
            scored.append((chunk, count))
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:top_k]

all_chunks = []
for doc in DOCUMENTS:
    for i, chunk in enumerate(chunk_by_paragraphs(doc["text"])):
        all_chunks.append({"doc_id": doc["id"], "title": doc["title"], "chunk_index": i, "text": chunk})

# "find" doesn't appear literally in any document, but it maps to the same
# "retrieval" concept as "retrieves" -- so semantic search can match the
# RAG documents even though keyword search has nothing to go on for that
# word. "database" and "meaning" DO appear literally, so keyword search
# still finds the vector database chunks.
query = "find database meaning"
print("Keyword-only:")
keyword_results = keyword_search_chunks(query, all_chunks, top_k=3)
if keyword_results:
    for chunk, score in keyword_results:
        print(f"  {chunk['title']} chunk {chunk['chunk_index']} (matched {score})")
else:
    print("  (no results)")

print("\nSemantic-only:")
for r in retriever.retrieve(query, top_k=4):
    print(f"  {r['metadata']['title']} chunk {r['metadata']['chunk_index']} (similarity {r['similarity']:.3f})")

print("\nNotice semantic search ranks the RAG chunks as relevant (through the shared 'retrieval' concept behind 'find'/'retrieves'), something keyword search has no way to detect since 'find' never appears in those documents at all.")

## Cell 7: Practical retriever example

A retriever built for real use needs to handle edge cases: empty queries, no good
matches, and confidence scoring instead of raw numbers.

In [ ]:
def confidence_label(similarity):
    if similarity >= 0.7:
        return "high"
    if similarity >= 0.4:
        return "medium"
    return "low"

def practical_retrieve(query, retriever, min_similarity=0.1):
    query = (query or "").strip()
    if not query:
        return {"status": "empty_query", "results": []}
    results = [r for r in retriever.retrieve(query, top_k=3) if r["similarity"] >= min_similarity]
    if not results:
        return {"status": "no_confident_match", "results": []}
    return {"status": "ok", "results": [{**r, "confidence": confidence_label(r["similarity"])} for r in results]}

for q in ["What is RAG?", "What's the weather today?", ""]:
    response = practical_retrieve(q, retriever)
    print(f"Query: {q!r} -> status: {response['status']}")
    for r in response["results"]:
        print(f"    {r['metadata']['title']} (similarity {r['similarity']:.3f}, {r['confidence']} confidence)")

## Cell 8: Integration with ranking

Retrieval finds candidates by similarity. Ranking can adjust the final order using
other signals too, like recency.

In [ ]:
raw_results = retriever.retrieve("What is RAG?", top_k=3)
recency_scores = {2: 0.2, 5: 0.95}

print("Raw order (by similarity):")
for r in raw_results:
    print(f"  {r['metadata']['title']} (similarity {r['similarity']:.3f})")

# A strong boost weight (0.5) on purpose, so the demo clearly shows a
# recency signal actually reordering the top result -- a weaker boost
# would just nudge scores without changing which document wins.
boost_weight = 0.5
boosted = []
for r in raw_results:
    recency = recency_scores.get(r["metadata"]["doc_id"], 0.5)
    final = r["similarity"] * (1 - boost_weight) + recency * boost_weight
    boosted.append({**r, "final_score": final})
boosted.sort(key=lambda r: r["final_score"], reverse=True)

print("\nRe-ranked with a recency boost:")
for r in boosted:
    print(f"  {r['metadata']['title']} (similarity {r['similarity']:.3f}, final {r['final_score']:.3f})")

## Cell 9: Key takeaways

- Retrieval is the bridge between a pile of documents and an LLM that can answer
  questions about them -- it decides whether the system even has a chance of
  getting the answer right.
- Query preprocessing (cleaning, expanding) can recover matches a raw query misses.
- Retrieval quality is measurable: precision@K, recall@K, and NDCG turn a vague
  impression into a number you can track and improve.
- Multi-query retrieval trades extra search cost for better recall on ambiguous or
  awkwardly-phrased questions.
- Hybrid retrieval (keyword + semantic) catches documents that either method alone
  would miss.
- A practical retriever needs to handle edge cases gracefully: empty queries, no
  good matches, confidence scoring -- not just the happy path.
- Retrieval finds a good POOL of candidates. Ranking (Day 6) decides the best final
  ORDER using similarity plus other signals that matter for your use case.